In [1]:
import chromadb
chroma_client = chromadb.Client()

In [2]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L12-v2",
    device="cuda",
    normalize_embeddings=True
)

print("SenTransformer Ready")

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2883.35it/s]


SenTransformer Ready


In [3]:
collection = chroma_client.get_or_create_collection(
    name="sen_transformer_top_10",
    embedding_function=sentence_transformer_ef
)

In [4]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
print("Data set yüklendi")

Data set yüklendi


In [5]:
doc_texts = []
doc_ids = []

for doc in dataset.docs_iter():
    doc_texts.append(doc.text)
    doc_ids.append(doc.doc_id)

### Vector Database Storage

In [6]:
from tqdm import tqdm

BATCH_SIZE = 5000
total_doc = len(doc_texts)

for i in tqdm(range(0, total_doc, BATCH_SIZE)):
    batch_texts = doc_texts[i: i + BATCH_SIZE]
    batch_ids = doc_ids[i: i + BATCH_SIZE]

    collection.add(
        documents=batch_texts,
        ids=batch_ids
    )

print("Vectorization!")

100%|██████████| 74/74 [1:07:52<00:00, 55.04s/it] 

Vectorization!


In [7]:
queries = []

for query in dataset.queries_iter():
    queries.append(query.text)

In [8]:
results = collection.query(
    query_texts=queries,
    n_results=10
)

In [9]:
results['ids']

[['516871',
  '647478',
  '2129172',
  '806300',
  '2357011',
  '1222510',
  '806263',
  '2433706',
  '708880',
  '775424'],
 ['188629',
  '251586',
  '2397950',
  '2061524',
  '272146',
  '1948304',
  '2437665',
  '1278316',
  '1614942',
  '1363774'],
 ['13898',
  '785960',
  '1382760',
  '941146',
  '802532',
  '1048798',
  '1989378',
  '1874948',
  '2031249',
  '462902'],
 ['316959',
  '562691',
  '1326263',
  '2156815',
  '687712',
  '42143',
  '1639962',
  '1469362',
  '729083',
  '1422464'],
 ['515031',
  '706326',
  '780987',
  '832086',
  '842375',
  '860147',
  '828931',
  '833668',
  '754830',
  '823332'],
 ['2267777',
  '421974',
  '1630942',
  '1209576',
  '2322923',
  '152040',
  '116684',
  '2117771',
  '1411855',
  '123783'],
 ['4332',
  '615358',
  '1712050',
  '2361455',
  '1297050',
  '1878340',
  '1438336',
  '2342190',
  '774950',
  '1913962'],
 ['9591',
  '143444',
  '1898638',
  '595391',
  '1445185',
  '2378793',
  '1472431',
  '494103',
  '2021787',
  '863960'],

In [13]:
query_ids = [query.query_id for query in dataset.queries_iter()]
query_ids

['123839',
 '188629',
 '13898',
 '316959',
 '515031',
 '123783',
 '4332',
 '9591',
 '114982',
 '563603',
 '104206',
 '1580851',
 '23678',
 '37480',
 '109454',
 '544250',
 '12519',
 '115015',
 '84287',
 '24998',
 '11925',
 '643',
 '113075',
 '1250387',
 '72012',
 '167865',
 '1425120',
 '24883',
 '27200',
 '136856',
 '6031',
 '1313191',
 '187115',
 '84110',
 '208234',
 '6431',
 '10799',
 '1417144',
 '32512',
 '677637',
 '97937',
 '113765',
 '12900',
 '1294909',
 '73745',
 '11692',
 '22173',
 '76662',
 '164301',
 '23344',
 '9367',
 '592217',
 '98546',
 '98175',
 '15189',
 '1256',
 '1815937',
 '1249076',
 '80274',
 '38489',
 '11311',
 '4754',
 '257711',
 '141668',
 '100895',
 '32945',
 '112321',
 '133725',
 '424464',
 '523456',
 '1327515',
 '12101',
 '9739',
 '189869',
 '143444',
 '78940',
 '12527',
 '327199',
 '7707',
 '97455',
 '1262433',
 '488875',
 '74026',
 '358108',
 '95179',
 '7862',
 '151797',
 '92980',
 '100876',
 '238228',
 '107755',
 '32539',
 '12953',
 '24843',
 '119528',
 '307

In [14]:
from collections import defaultdict
sen_transformer_top_10 = defaultdict(list)

for i in range(len(query_ids)):
    sen_transformer_top_10[query_ids[i]] = results['ids'][i]

sen_transformer_top_10 = dict(sen_transformer_top_10)

In [15]:
sen_transformer_top_10

{'123839': ['516871',
  '647478',
  '2129172',
  '806300',
  '2357011',
  '1222510',
  '806263',
  '2433706',
  '708880',
  '775424'],
 '188629': ['188629',
  '251586',
  '2397950',
  '2061524',
  '272146',
  '1948304',
  '2437665',
  '1278316',
  '1614942',
  '1363774'],
 '13898': ['13898',
  '785960',
  '1382760',
  '941146',
  '802532',
  '1048798',
  '1989378',
  '1874948',
  '2031249',
  '462902'],
 '316959': ['316959',
  '562691',
  '1326263',
  '2156815',
  '687712',
  '42143',
  '1639962',
  '1469362',
  '729083',
  '1422464'],
 '515031': ['515031',
  '706326',
  '780987',
  '832086',
  '842375',
  '860147',
  '828931',
  '833668',
  '754830',
  '823332'],
 '123783': ['2267777',
  '421974',
  '1630942',
  '1209576',
  '2322923',
  '152040',
  '116684',
  '2117771',
  '1411855',
  '123783'],
 '4332': ['4332',
  '615358',
  '1712050',
  '2361455',
  '1297050',
  '1878340',
  '1438336',
  '2342190',
  '774950',
  '1913962'],
 '9591': ['9591',
  '143444',
  '1898638',
  '595391',
 

In [16]:
len(sen_transformer_top_10.keys())

1444

In [19]:
len(sen_transformer_top_10["13898"])

10

In [20]:
import pandas as pd
df_sen_transformer = pd.DataFrame(sen_transformer_top_10)

In [21]:
df_sen_transformer

,123839,188629,13898,316959,515031,123783,4332,9591,114982,563603,...,37970,2612,66818,862773,11326,896124,12319,4421,296526,341793
0,516871,188629,13898,316959,515031,2267777,4332,9591,725799,251631,...,37970,746818,66818,862773,1161272,747395,891422,1786733,1633820,341793
1,647478,251586,785960,562691,706326,421974,615358,143444,1454057,12846,...,580876,1137023,386220,1426706,11326,2054921,12319,1297589,718007,1857381
2,2129172,2397950,1382760,1326263,780987,1630942,1712050,1898638,1478113,273547,...,177545,2612,287026,865851,1517411,573381,276196,1111917,1901019,576283
3,806300,2061524,941146,2156815,832086,1209576,2361455,595391,2304325,563603,...,484539,1084496,2387535,2272303,623784,538460,2073704,365980,420896,2303264
4,2357011,272146,802532,687712,842375,2322923,1297050,1445185,294327,840695,...,736630,1963277,532111,189807,1636981,1038074,1119295,1554833,296526,576336
5,1222510,1948304,1048798,42143,860147,152040,1878340,2378793,2352224,2091601,...,154939,1089914,242676,1964415,1210930,561358,2086621,1224508,128653,2197644
6,806263,2437665,1989378,1639962,828931,116684,1438336,1472431,95752,2307963,...,208832,348197,309952,319262,1207519,2253573,1670681,1584239,1901020,1174734
7,2433706,1278316,1874948,1469362,833668,2117771,2342190,494103,2352254,2146572,...,2313722,1304252,1376367,2126545,272778,896124,373696,2271988,194498,876840
8,708880,1614942,2031249,729083,754830,1411855,774950,2021787,102670,2152054,...,1110574,858691,889006,275965,1901218,2121565,173798,2283929,1438397,576278
9,775424,1363774,462902,1422464,823332,123783,1913962,863960,2298231,1674515,...,718515,353802,775005,1653529,653832,2331828,2110002,1227506,1265304,355461


In [22]:
df_sen_transformer.to_parquet("sen_transformer_top_10.parquet")